In [12]:
import sys
import pandas as pd

from PySide6.QtWidgets import (
    QApplication, QDialog, QVBoxLayout, QLabel, QTableWidget, QTableWidgetItem,
    QPushButton, QMessageBox
)
from PySide6.QtCore import Qt


NEEDED_COLS = ["수취인명", "상품명", "옵션명", "샵플링 매핑옵션코드", "주문수량", "실재고", "자사코드"]


def load_and_check_stock(excel_path: str):
    """
    엑셀 로직 그대로:
    - (R역할) 같은 '샵플링 매핑옵션코드'별 주문수량 합계
    - (H역할) 실재고 >= 합계면 출고 가능, 아니면 재고 부족
    반환:
      df_all: 전체 df + '총주문수량' + '결과값'
      df_lack: 재고 부족인 행들(팝업에 띄울 목록)
    """
    df = pd.read_excel(excel_path, dtype=str)

    # 컬럼 누락 방지
    missing = [c for c in NEEDED_COLS if c not in df.columns]
    if missing:
        raise ValueError(f"엑셀에 필요한 컬럼이 없습니다: {missing}")

    # 공백 정리
    for c in NEEDED_COLS:
        df[c] = df[c].astype(str).replace("nan", "").str.strip()

    # 숫자 변환(주문수량/실재고)
    df["주문수량_num"] = pd.to_numeric(df["주문수량"], errors="coerce").fillna(0).astype(int)
    df["실재고_num"] = pd.to_numeric(df["실재고"], errors="coerce").fillna(0).astype(int)

    # 유효행 기준:
    # - 수취인명 있는 행만
    # - 매핑옵션코드가 비어있거나 0이면 제외(샘플에 0이 꽤 있음)
    df["유효"] = (df["수취인명"] != "") & (df["샵플링 매핑옵션코드"] != "") & (df["샵플링 매핑옵션코드"] != "0")

    # (R 역할) 옵션코드별 총 주문수량
    df["총주문수량"] = 0
    df.loc[df["유효"], "총주문수량"] = (
        df.loc[df["유효"]]
        .groupby("샵플링 매핑옵션코드")["주문수량_num"]
        .transform("sum")
        .astype(int)
    )

    # (H 역할) 결과값
    df["결과값"] = ""
    df.loc[df["유효"], "결과값"] = df.loc[df["유효"]].apply(
        lambda r: "출고 가능" if r["실재고_num"] >= r["총주문수량"] else "재고 부족",
        axis=1
    )

    # 팝업에 보여줄 "재고 부족" 목록
    df_lack = df[(df["유효"]) & (df["결과값"] == "재고 부족")].copy()

    # 표시용 컬럼만 남기기 (네가 원하는 6개)
    show_cols = ["수취인명", "상품명", "옵션명", "주문수량", "실재고", "자사코드"]
    df_lack_show = df_lack[show_cols].copy()

    return df, df_lack_show


class LackStockDialog(QDialog):
    def __init__(self, df_lack: pd.DataFrame, parent=None):
        super().__init__(parent)
        self.setWindowTitle("재고 부족")
        self.resize(900, 500)

        layout = QVBoxLayout(self)

        title = QLabel("실재고가 부족합니다")
        title.setAlignment(Qt.AlignLeft)
        title.setStyleSheet("font-size: 18px; font-weight: 700;")
        layout.addWidget(title)

        sub = QLabel("아래 주문건을 확인하세요.")
        sub.setStyleSheet("color: #555;")
        layout.addWidget(sub)

        self.table = QTableWidget(self)
        self.table.setColumnCount(len(df_lack.columns))
        self.table.setHorizontalHeaderLabels(df_lack.columns.tolist())
        self.table.setRowCount(len(df_lack))

        # 채우기
        for r in range(len(df_lack)):
            for c, col in enumerate(df_lack.columns):
                val = "" if pd.isna(df_lack.iat[r, c]) else str(df_lack.iat[r, c])
                item = QTableWidgetItem(val)
                item.setFlags(item.flags() ^ Qt.ItemIsEditable)
                self.table.setItem(r, c, item)

        self.table.resizeColumnsToContents()
        self.table.setSortingEnabled(True)
        layout.addWidget(self.table)

        btn = QPushButton("확인")
        btn.clicked.connect(self.accept)
        layout.addWidget(btn)


def main():
    
    excel_path = r"C:\Users\ckm00\Downloads\test.xlsx"

    df_all, df_lack = load_and_check_stock(excel_path)

    # ✅ 이미 만들어진 QApplication이 있으면 재사용
    app = QApplication.instance() or QApplication(sys.argv)

    if df_lack.empty:
        QMessageBox.information(None, "재고 확인", "재고 부족 주문건이 없습니다. (출고 가능)")
        return

    dlg = LackStockDialog(df_lack)
    dlg.exec()


if __name__ == "__main__":
    main()